In [1]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
device = 0 if torch.cuda.is_available() else -1

print(device)

Torch: 2.10.0+cu130
CUDA available: True
CUDA version: 13.0
0


In [15]:
from transformers import pipeline

pipe = pipeline("fill-mask", top_k=5)

pipe("The capital of France is <mask>.")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.2703714668750763,
  'token': 2201,
  'token_str': ' Paris',
  'sequence': 'The capital of France is Paris.'},
 {'score': 0.05588307976722717,
  'token': 12790,
  'token_str': ' Lyon',
  'sequence': 'The capital of France is Lyon.'},
 {'score': 0.029897993430495262,
  'token': 4612,
  'token_str': ' Barcelona',
  'sequence': 'The capital of France is Barcelona.'},
 {'score': 0.023081643506884575,
  'token': 12696,
  'token_str': ' Monaco',
  'sequence': 'The capital of France is Monaco.'},
 {'score': 0.0209799911826849,
  'token': 5459,
  'token_str': ' Berlin',
  'sequence': 'The capital of France is Berlin.'}]

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert/distilbert-base-uncased").to(device)

text = "The cat sat on the [MASK] located inside teh [MASK]."
inputs = tokenizer(text, return_tensors="pt").to(device)

# Forward pass (no generate() for MLMs)
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

# Find mask position
mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

pred_id = logits[0, mask_idx].argmax(dim=-1)

print(tokenizer.decode(pred_id))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

bench palace


In [16]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")
model = AutoModelForMaskedLM.from_pretrained("FacebookAI/roberta-base").to(device)

text = "Fill in the blank: The capital of France is <mask> and it is in the continent of <mask>."
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

pred_id = logits[0, mask_idx].argmax(dim=-1)

print(tokenizer.decode(pred_id))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Paris Africa


# Comparing two models

In [17]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM

models = ["google-bert/bert-base-uncased", "distilbert/distilbert-base-uncased"]

text = "The capital of France is [MASK]."

for model_name in models:
    print(f"\nModel: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits  # (1, seq_len, vocab_size)
    
    # find mask position
    mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    
    mask_logits = logits[0, mask_idx, :]  # (1, vocab_size)
    
    probs = F.softmax(mask_logits, dim=-1)
    
    # top 5
    topk = torch.topk(probs, k=5)
    
    for i in range(5):
        token_id = topk.indices[0][i]
        token = tokenizer.decode([token_id])
        prob = topk.values[0][i].item()
        print(f"{token} → {prob:.4f}")


Model: google-bert/bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush\.cache\huggingface\hub\models--google-bert--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


paris → 0.4168
lille → 0.0714
lyon → 0.0634
marseille → 0.0444
tours → 0.0303

Model: distilbert/distilbert-base-uncased


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

marseille → 0.1427
nantes → 0.0902
toulouse → 0.0881
paris → 0.0862
lyon → 0.0772
